# CSQA Fixed-Site Rescue Patching And Activation Steering Analysis

Intervention notebook for rescue-site validation with a single late-layer intervention location.

- model family: Qwen2.5
- target checkpoint: 3B instruct model
- corruption type: additive noise on one earlier decoder layer output
- patch type: same-example clean activation replacement at one fixed rescue layer
- steering types:
  - contrastive mean direction
  - probe-normal direction
- decision space: constrained A-E answer-choice logits only


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.data.load_csqa import load_csqa

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)


## Configuration

Global study settings only.

- RESCUE_LAYER_NUMBER: fixed intervention layer used for clean patching and steering
- CORRUPTION_LAYER_NUMBERS = None: all layers before the rescue layer are corrupted one at a time
- CORRUPTION_NOISE_SCALES: corruption strengths
- STEERING_SCALES: additive steering strengths


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
TRAIN_SPLIT = "train"
EVAL_SPLIT = "validation"
TRAIN_LIMIT = 512
EVAL_LIMIT = None
CORRUPTION_NOISE_SCALES = [0.10, 0.25, 0.50]
RESCUE_LAYER_NUMBER = 28
CORRUPTION_LAYER_NUMBERS = None
STEERING_SCALES = [0.5, 1.0, 2.0]
SEED = 42


## Environment Setup


In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

LETTERS = ["A", "B", "C", "D", "E"]
MAX_SEQ_LEN = 384
CLEAN_BATCH_SIZE = 4
INTERVENTION_BATCH_SIZE = 2

train_rows = load_csqa(split=TRAIN_SPLIT, limit=TRAIN_LIMIT).copy()
eval_rows = load_csqa(split=EVAL_SPLIT, limit=EVAL_LIMIT).copy()

for frame in [train_rows, eval_rows]:
    frame["n_choices"] = frame["csqa_choices"].map(len)
    frame["prompt_len_chars"] = frame["text"].str.len()

if torch.cuda.is_available():
    if torch.cuda.is_bf16_supported():
        model_dtype = torch.bfloat16
    else:
        model_dtype = torch.float16
    device_map = "auto"
else:
    model_dtype = torch.float32
    device_map = None

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=model_dtype,
    device_map=device_map,
    attn_implementation="eager",
)
model.eval()

input_device = model.device if hasattr(model, "device") else next(model.parameters()).device


def build_answer_token_ids(tok):
    out = {}
    for letter in LETTERS:
        ids = tok(" " + letter, add_special_tokens=False)["input_ids"]
        if len(ids) != 1:
            raise ValueError(f"Answer token '{letter}' is not single-token: {ids}")
        out[letter] = int(ids[0])
    return out


answer_token_ids = build_answer_token_ids(tok)
answer_ids = [answer_token_ids[l] for l in LETTERS]
answer_id_tensor = torch.tensor(answer_ids, dtype=torch.long)

display(eval_rows[["example_id", "answerKey", "prompt_len_chars"]].head())
print("train rows:", len(train_rows))
print("eval rows:", len(eval_rows))
print("answer token ids:", answer_token_ids)


## Helper Functions


In [ ]:
def get_decoder_layers(model):
    candidates = [
        "model.layers",
        "transformer.h",
        "gpt_neox.layers",
    ]
    for path in candidates:
        cur = model
        ok = True
        for part in path.split("."):
            if not hasattr(cur, part):
                ok = False
                break
            cur = getattr(cur, part)
        if ok:
            return cur
    raise ValueError("Could not locate decoder layers on this model.")


def encode_prompts(texts, tok, max_seq_len):
    batch = tok(
        list(texts),
        add_special_tokens=False,
        truncation=True,
        max_length=max_seq_len,
        padding=True,
        return_tensors="pt",
    )
    pos = []
    for mask in batch["attention_mask"]:
        nz = torch.nonzero(mask, as_tuple=False).view(-1)
        pos.append(int(nz[-1].item()))
    batch["decision_pos"] = torch.tensor(pos, dtype=torch.long)
    return batch


def unpack_output_hidden(output):
    if isinstance(output, tuple):
        return output[0]
    return output


def repack_output_hidden(output, new_hidden):
    if isinstance(output, tuple):
        return (new_hidden,) + tuple(output[1:])
    return new_hidden


def select_choice_logits(logits, decision_pos):
    row_idx = torch.arange(logits.shape[0], device=logits.device)
    answer_ids_on_device = answer_id_tensor.to(logits.device)
    return logits[row_idx, decision_pos][:, answer_ids_on_device].float()


def compute_choice_metrics(choice_logits, true_choice_idx):
    predicted_answer_index = choice_logits.argmax(dim=-1)
    row_idx = torch.arange(choice_logits.shape[0], device=choice_logits.device)
    true_logits = choice_logits[row_idx, true_choice_idx]
    other_logits = choice_logits.clone()
    other_logits[row_idx, true_choice_idx] = -torch.inf
    highest_other_logit = other_logits.max(dim=-1).values
    true_answer_vs_best_other_logit_gap = true_logits - highest_other_logit
    return predicted_answer_index.detach().cpu(), true_answer_vs_best_other_logit_gap.detach().cpu()


def apply_token_noise(hidden, decision_pos, noise_scale):
    row_idx = torch.arange(hidden.shape[0], device=hidden.device)
    token_hidden = hidden[row_idx, decision_pos]
    rms = token_hidden.float().pow(2).mean(dim=-1, keepdim=True).sqrt().to(token_hidden.dtype)
    noise = torch.randn_like(token_hidden) * (noise_scale * rms)
    hidden_out = hidden.clone()
    hidden_out[row_idx, decision_pos] = token_hidden + noise
    return hidden_out


def patch_token_hidden(hidden, decision_pos, clean_hidden):
    row_idx = torch.arange(hidden.shape[0], device=hidden.device)
    hidden_out = hidden.clone()
    hidden_out[row_idx, decision_pos] = clean_hidden.to(hidden.device, dtype=hidden.dtype)
    return hidden_out


def apply_token_steering(hidden, decision_pos, direction, steering_scale):
    row_idx = torch.arange(hidden.shape[0], device=hidden.device)
    token_hidden = hidden[row_idx, decision_pos]
    rms = token_hidden.float().pow(2).mean(dim=-1, keepdim=True).sqrt().to(token_hidden.dtype)
    direction = direction.to(hidden.device, dtype=hidden.dtype)
    hidden_out = hidden.clone()
    hidden_out[row_idx, decision_pos] = token_hidden + (steering_scale * rms) * direction.unsqueeze(0)
    return hidden_out


def set_all_seeds(seed):
    torch.manual_seed(int(seed))
    np.random.seed(int(seed) % (2**32 - 1))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


decoder_layers = get_decoder_layers(model)
L = len(decoder_layers)

rescue_layer_number = int(RESCUE_LAYER_NUMBER)

if CORRUPTION_LAYER_NUMBERS is None:
    corruption_layer_numbers = list(range(1, rescue_layer_number))
else:
    corruption_layer_numbers = [int(x) for x in CORRUPTION_LAYER_NUMBERS]

corruption_layer_numbers = sorted(set(corruption_layer_numbers))
rescue_layer_index = rescue_layer_number - 1
corruption_layer_indices = [x - 1 for x in corruption_layer_numbers]

print("decoder layers:", L)
print("rescue layer:", rescue_layer_number)
print("corruption layers:", corruption_layer_numbers[:12], "..." if len(corruption_layer_numbers) > 12 else "")


### Decision Score Used In All Summaries

	rue_answer_vs_best_other_logit_gap means:

- logit of the true answer choice
- minus the highest logit among the remaining A-E choices

Negative change means the true answer became less competitive.


## Data Generation And Extraction


### Clean Rescue-Layer Feature Extraction

Clean decoder layer outputs are extracted at the fixed rescue layer for steering direction construction and evaluation baselines.


In [ ]:
def extract_clean_layer_features(frame, target_layer_0based, batch_size, desc):
    rows = []
    hidden_blocks = []
    target_module = decoder_layers[target_layer_0based]

    for start in tqdm(range(0, len(frame), batch_size), total=int(math.ceil(len(frame) / batch_size)), desc=desc):
        batch_df = frame.iloc[start:start + batch_size].reset_index(drop=True)
        batch_cpu = encode_prompts(batch_df["text"], tok, MAX_SEQ_LEN)
        decision_pos = batch_cpu.pop("decision_pos")
        batch = {k: v.to(input_device) for k, v in batch_cpu.items()}
        decision_pos = decision_pos.to(input_device)
        true_choice_idx = torch.tensor([LETTERS.index(str(x)) for x in batch_df["answerKey"].tolist()], dtype=torch.long, device=input_device)

        captured = {}

        def capture_hook(module, inputs, output):
            captured["hidden"] = unpack_output_hidden(output).detach()

        handle = target_module.register_forward_hook(capture_hook)
        try:
            with torch.inference_mode():
                out = model(**batch, return_dict=True, use_cache=False)
        finally:
            handle.remove()

        choice_logits = select_choice_logits(out.logits, decision_pos)
        predicted_answer_index, true_answer_vs_best_other_logit_gap = compute_choice_metrics(choice_logits, true_choice_idx)
        clean_is_correct = predicted_answer_index.eq(true_choice_idx.detach().cpu())

        row_idx = torch.arange(len(batch_df), device=decision_pos.device)
        hidden = captured["hidden"][row_idx, decision_pos].detach().cpu().to(torch.float32)
        hidden_blocks.append(hidden)

        for i, row in batch_df.iterrows():
            rows.append(
                {
                    "example_id": row["example_id"],
                    "answerKey": row["answerKey"],
                    "clean_prediction_idx": int(predicted_answer_index[i].item()),
                    "clean_true_answer_vs_best_other_logit_gap": float(true_answer_vs_best_other_logit_gap[i].item()),
                    "clean_is_correct": bool(clean_is_correct[i].item()),
                }
            )

    return pd.DataFrame(rows), torch.cat(hidden_blocks, dim=0)


train_clean_df, train_rescue_hidden = extract_clean_layer_features(
    train_rows,
    rescue_layer_index,
    CLEAN_BATCH_SIZE,
    desc="train rescue-layer extraction",
)

eval_clean_df, eval_rescue_hidden = extract_clean_layer_features(
    eval_rows,
    rescue_layer_index,
    CLEAN_BATCH_SIZE,
    desc="eval rescue-layer extraction",
)

clean_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "n_examples": len(train_clean_df),
            "clean_accuracy": float(train_clean_df["clean_is_correct"].mean()),
            "mean_true_answer_vs_best_other_logit_gap": float(train_clean_df["clean_true_answer_vs_best_other_logit_gap"].mean()),
        },
        {
            "split": "validation",
            "n_examples": len(eval_clean_df),
            "clean_accuracy": float(eval_clean_df["clean_is_correct"].mean()),
            "mean_true_answer_vs_best_other_logit_gap": float(eval_clean_df["clean_true_answer_vs_best_other_logit_gap"].mean()),
        },
    ]
)

display(clean_summary.round(4))


### Steering Direction Construction

Two additive steering directions are constructed at the fixed rescue layer:

- contrastive mean direction: mean(correct) - mean(incorrect)
- probe-normal direction: linear probe normal for clean correctness classification

The first direction is a direct class contrast. The second direction is the separating normal learned by a linear classifier at the rescue layer.


In [ ]:
probe_train_epochs = 200
probe_train_learning_rate = 5e-2
probe_train_weight_decay = 1e-4


def build_contrastive_mean_direction(hidden_cache, is_correct_mask):
    mask = torch.as_tensor(is_correct_mask.to_numpy() if hasattr(is_correct_mask, "to_numpy") else is_correct_mask, dtype=torch.bool)
    if int(mask.sum().item()) == 0 or int((~mask).sum().item()) == 0:
        raise ValueError("Both correct and incorrect examples are required for contrastive mean direction construction.")

    correct_mean = hidden_cache[mask].mean(dim=0)
    incorrect_mean = hidden_cache[~mask].mean(dim=0)
    raw_direction = (correct_mean - incorrect_mean).to(torch.float32)
    raw_norm = float(raw_direction.norm().item())
    direction = raw_direction / raw_direction.norm().clamp_min(1e-12)
    projection = hidden_cache @ direction

    return direction, {
        "steering_method": "Contrastive mean direction",
        "n_examples": int(hidden_cache.shape[0]),
        "n_correct": int(mask.sum().item()),
        "n_incorrect": int((~mask).sum().item()),
        "raw_direction_norm": raw_norm,
        "mean_projection_correct": float(projection[mask].mean().item()),
        "mean_projection_incorrect": float(projection[~mask].mean().item()),
        "probe_train_accuracy": np.nan,
    }


def build_probe_normal_direction(hidden_cache, is_correct_mask):
    mask = torch.as_tensor(is_correct_mask.to_numpy() if hasattr(is_correct_mask, "to_numpy") else is_correct_mask, dtype=torch.bool)
    if int(mask.sum().item()) == 0 or int((~mask).sum().item()) == 0:
        raise ValueError("Both correct and incorrect examples are required for probe direction construction.")

    x = hidden_cache.to(torch.float32)
    y = mask.to(torch.float32).unsqueeze(-1)
    mu = x.mean(dim=0)
    sigma = x.std(dim=0).clamp_min(1e-6)
    xz = (x - mu) / sigma

    probe = torch.nn.Linear(xz.shape[1], 1)
    optimizer = torch.optim.AdamW(
        probe.parameters(),
        lr=probe_train_learning_rate,
        weight_decay=probe_train_weight_decay,
    )

    for _ in tqdm(range(probe_train_epochs), desc="probe training"):
        optimizer.zero_grad(set_to_none=True)
        logits = probe(xz)
        loss = F.binary_cross_entropy_with_logits(logits, y)
        loss.backward()
        optimizer.step()

    with torch.inference_mode():
        logits = probe(xz)
        preds = logits.squeeze(-1).ge(0.0)
        train_accuracy = float(preds.eq(mask).float().mean().item())
        raw_weight = probe.weight.detach().cpu().squeeze(0).to(torch.float32)

    raw_direction = raw_weight / sigma.cpu()
    projection = x @ raw_direction
    if projection[mask].mean() < projection[~mask].mean():
        raw_direction = -raw_direction
        projection = -projection

    raw_norm = float(raw_direction.norm().item())
    direction = raw_direction / raw_direction.norm().clamp_min(1e-12)

    return direction, {
        "steering_method": "Probe-normal direction",
        "n_examples": int(x.shape[0]),
        "n_correct": int(mask.sum().item()),
        "n_incorrect": int((~mask).sum().item()),
        "raw_direction_norm": raw_norm,
        "mean_projection_correct": float(projection[mask].mean().item()),
        "mean_projection_incorrect": float(projection[~mask].mean().item()),
        "probe_train_accuracy": train_accuracy,
    }


contrastive_direction, contrastive_info = build_contrastive_mean_direction(
    train_rescue_hidden,
    train_clean_df["clean_is_correct"],
)

probe_direction, probe_info = build_probe_normal_direction(
    train_rescue_hidden,
    train_clean_df["clean_is_correct"],
)

steering_direction_info_df = pd.DataFrame([contrastive_info, probe_info])
steering_directions = {
    "Contrastive mean direction": contrastive_direction,
    "Probe-normal direction": probe_direction,
}

display(steering_direction_info_df.round(4))


### Fixed-Site Corruption, Same-Example Clean Patching, And Steering

Each earlier corruption site is scanned one at a time across multiple configured corruption scales. The rescue site is fixed. Three intervention regimes are evaluated:

- corruption only
- same-example clean patch at the rescue layer
- additive steering at the rescue layer


In [ ]:
def run_fixed_site_intervention_scan(
    frame,
    corruption_layers_0based,
    rescue_layer_0based,
    noise_scale,
    batch_size,
    intervention_kind,
    clean_rescue_hidden=None,
    steering_direction=None,
    steering_scale=None,
    desc="fixed-site intervention",
):
    rows = []
    rescue_module = decoder_layers[rescue_layer_0based]

    for corruption_layer_0based in tqdm(corruption_layers_0based, desc=desc):
        corruption_module = decoder_layers[corruption_layer_0based]

        for start in range(0, len(frame), batch_size):
            batch_df = frame.iloc[start:start + batch_size].reset_index(drop=True)
            batch_cpu = encode_prompts(batch_df["text"], tok, MAX_SEQ_LEN)
            decision_pos = batch_cpu.pop("decision_pos")
            batch = {k: v.to(input_device) for k, v in batch_cpu.items()}
            decision_pos = decision_pos.to(input_device)
            true_choice_idx = torch.tensor([LETTERS.index(str(x)) for x in batch_df["answerKey"].tolist()], dtype=torch.long, device=input_device)

            local_seed = int(SEED + 10007 * (corruption_layer_0based + 1) + 97 * start)
            set_all_seeds(local_seed)
            handles = []

            def corruption_hook(module, inputs, output):
                hidden = unpack_output_hidden(output)
                hidden = apply_token_noise(hidden, decision_pos, noise_scale)
                return repack_output_hidden(output, hidden)

            handles.append(corruption_module.register_forward_hook(corruption_hook))

            if intervention_kind == "clean_patch":
                batch_clean_hidden = clean_rescue_hidden[start:start + len(batch_df)]

                def patch_hook(module, inputs, output):
                    hidden = unpack_output_hidden(output)
                    hidden = patch_token_hidden(hidden, decision_pos, batch_clean_hidden)
                    return repack_output_hidden(output, hidden)

                handles.append(rescue_module.register_forward_hook(patch_hook))

            elif intervention_kind == "steering":
                if steering_direction is None or steering_scale is None:
                    raise ValueError("steering_direction and steering_scale are required for steering scans.")

                def steering_hook(module, inputs, output):
                    hidden = unpack_output_hidden(output)
                    hidden = apply_token_steering(hidden, decision_pos, steering_direction, steering_scale)
                    return repack_output_hidden(output, hidden)

                handles.append(rescue_module.register_forward_hook(steering_hook))

            try:
                with torch.inference_mode():
                    out = model(**batch, return_dict=True, use_cache=False)
            finally:
                for handle in handles:
                    handle.remove()

            choice_logits = select_choice_logits(out.logits, decision_pos)
            predicted_answer_index, true_answer_vs_best_other_logit_gap = compute_choice_metrics(choice_logits, true_choice_idx)
            is_correct = predicted_answer_index.eq(true_choice_idx.detach().cpu())

            for i, row in batch_df.iterrows():
                rows.append(
                    {
                        "example_id": row["example_id"],
                        "corruption_layer": int(corruption_layer_0based),
                        "predicted_answer_index": int(predicted_answer_index[i].item()),
                        "true_answer_vs_best_other_logit_gap": float(true_answer_vs_best_other_logit_gap[i].item()),
                        "is_correct": bool(is_correct[i].item()),
                    }
                )

    return pd.DataFrame(rows)


def run_clean_steering_control(
    frame,
    rescue_layer_0based,
    batch_size,
    steering_direction,
    steering_scale,
    desc="clean steering control",
):
    rows = []
    rescue_module = decoder_layers[rescue_layer_0based]

    for start in tqdm(range(0, len(frame), batch_size), total=int(math.ceil(len(frame) / batch_size)), desc=desc):
        batch_df = frame.iloc[start:start + batch_size].reset_index(drop=True)
        batch_cpu = encode_prompts(batch_df["text"], tok, MAX_SEQ_LEN)
        decision_pos = batch_cpu.pop("decision_pos")
        batch = {k: v.to(input_device) for k, v in batch_cpu.items()}
        decision_pos = decision_pos.to(input_device)
        true_choice_idx = torch.tensor([LETTERS.index(str(x)) for x in batch_df["answerKey"].tolist()], dtype=torch.long, device=input_device)

        def steering_hook(module, inputs, output):
            hidden = unpack_output_hidden(output)
            hidden = apply_token_steering(hidden, decision_pos, steering_direction, steering_scale)
            return repack_output_hidden(output, hidden)

        handle = rescue_module.register_forward_hook(steering_hook)
        try:
            with torch.inference_mode():
                out = model(**batch, return_dict=True, use_cache=False)
        finally:
            handle.remove()

        choice_logits = select_choice_logits(out.logits, decision_pos)
        predicted_answer_index, true_answer_vs_best_other_logit_gap = compute_choice_metrics(choice_logits, true_choice_idx)
        is_correct = predicted_answer_index.eq(true_choice_idx.detach().cpu())

        for i, row in batch_df.iterrows():
            rows.append(
                {
                    "example_id": row["example_id"],
                    "predicted_answer_index": int(predicted_answer_index[i].item()),
                    "true_answer_vs_best_other_logit_gap": float(true_answer_vs_best_other_logit_gap[i].item()),
                    "is_correct": bool(is_correct[i].item()),
                }
            )

    return pd.DataFrame(rows)


corrupted_frames = []
for corruption_noise_scale in CORRUPTION_NOISE_SCALES:
    corrupted_part = run_fixed_site_intervention_scan(
        eval_rows,
        corruption_layer_indices,
        rescue_layer_index,
        float(corruption_noise_scale),
        INTERVENTION_BATCH_SIZE,
        intervention_kind="corruption_only",
        desc=f"corruption only, scale {corruption_noise_scale}",
    ).rename(
        columns={
            "predicted_answer_index": "corrupted_prediction_idx",
            "true_answer_vs_best_other_logit_gap": "corrupted_true_answer_vs_best_other_logit_gap",
            "is_correct": "corrupted_is_correct",
        }
    )
    corrupted_part["corruption_noise_scale"] = float(corruption_noise_scale)
    corrupted_frames.append(corrupted_part)

corrupted_df = pd.concat(corrupted_frames, ignore_index=True)
corrupted_df = corrupted_df.merge(
    eval_clean_df[["example_id", "clean_prediction_idx", "clean_true_answer_vs_best_other_logit_gap", "clean_is_correct"]],
    on="example_id",
    how="left",
    validate="many_to_one",
)
corrupted_df["corruption_layer_number"] = corrupted_df["corruption_layer"] + 1
corrupted_df["delta_true_answer_vs_best_other_logit_gap"] = corrupted_df["corrupted_true_answer_vs_best_other_logit_gap"] - corrupted_df["clean_true_answer_vs_best_other_logit_gap"]
corrupted_df["prediction_preserved"] = corrupted_df["corrupted_prediction_idx"].eq(corrupted_df["clean_prediction_idx"])
corrupted_df["clean_correct_broken"] = corrupted_df["clean_is_correct"] & (~corrupted_df["corrupted_is_correct"])

patch_frames = []
for corruption_noise_scale in CORRUPTION_NOISE_SCALES:
    patch_part = run_fixed_site_intervention_scan(
        eval_rows,
        corruption_layer_indices,
        rescue_layer_index,
        float(corruption_noise_scale),
        INTERVENTION_BATCH_SIZE,
        intervention_kind="clean_patch",
        clean_rescue_hidden=eval_rescue_hidden,
        desc=f"clean patch, scale {corruption_noise_scale}",
    ).rename(
        columns={
            "predicted_answer_index": "patched_prediction_idx",
            "true_answer_vs_best_other_logit_gap": "patched_true_answer_vs_best_other_logit_gap",
            "is_correct": "patched_is_correct",
        }
    )
    patch_part["corruption_noise_scale"] = float(corruption_noise_scale)
    patch_frames.append(patch_part)

patch_df = pd.concat(patch_frames, ignore_index=True)
patch_df = patch_df.merge(
    corrupted_df[[
        "example_id",
        "corruption_layer",
        "corruption_layer_number",
        "corruption_noise_scale",
        "clean_prediction_idx",
        "clean_true_answer_vs_best_other_logit_gap",
        "clean_is_correct",
        "corrupted_prediction_idx",
        "corrupted_true_answer_vs_best_other_logit_gap",
        "corrupted_is_correct",
        "clean_correct_broken",
    ]],
    on=["example_id", "corruption_layer", "corruption_noise_scale"],
    how="left",
    validate="one_to_one",
)
patch_df["rescue_layer_number"] = rescue_layer_number
patch_df["recovered_true_answer_vs_best_other_logit_gap"] = patch_df["patched_true_answer_vs_best_other_logit_gap"] - patch_df["corrupted_true_answer_vs_best_other_logit_gap"]
patch_df["prediction_matches_clean"] = patch_df["patched_prediction_idx"].eq(patch_df["clean_prediction_idx"])
patch_df["broken_case_rescued"] = patch_df["clean_correct_broken"] & patch_df["patched_is_correct"]

steering_frames = []
for steering_method, steering_direction in steering_directions.items():
    for steering_scale in STEERING_SCALES:
        for corruption_noise_scale in CORRUPTION_NOISE_SCALES:
            steering_part = run_fixed_site_intervention_scan(
                eval_rows,
                corruption_layer_indices,
                rescue_layer_index,
                float(corruption_noise_scale),
                INTERVENTION_BATCH_SIZE,
                intervention_kind="steering",
                steering_direction=steering_direction,
                steering_scale=steering_scale,
                desc=f"{steering_method}, steering scale {steering_scale}, corruption scale {corruption_noise_scale}",
            ).rename(
                columns={
                    "predicted_answer_index": "steered_prediction_idx",
                    "true_answer_vs_best_other_logit_gap": "steered_true_answer_vs_best_other_logit_gap",
                    "is_correct": "steered_is_correct",
                }
            )
            steering_part["steering_method"] = steering_method
            steering_part["steering_scale"] = float(steering_scale)
            steering_part["corruption_noise_scale"] = float(corruption_noise_scale)
            steering_frames.append(steering_part)

steering_df = pd.concat(steering_frames, ignore_index=True)
steering_df = steering_df.merge(
    corrupted_df[[
        "example_id",
        "corruption_layer",
        "corruption_layer_number",
        "corruption_noise_scale",
        "clean_prediction_idx",
        "clean_true_answer_vs_best_other_logit_gap",
        "clean_is_correct",
        "corrupted_prediction_idx",
        "corrupted_true_answer_vs_best_other_logit_gap",
        "corrupted_is_correct",
        "clean_correct_broken",
    ]],
    on=["example_id", "corruption_layer", "corruption_noise_scale"],
    how="left",
    validate="many_to_one",
)
steering_df["rescue_layer_number"] = rescue_layer_number
steering_df["recovered_true_answer_vs_best_other_logit_gap"] = steering_df["steered_true_answer_vs_best_other_logit_gap"] - steering_df["corrupted_true_answer_vs_best_other_logit_gap"]
steering_df["prediction_matches_clean"] = steering_df["steered_prediction_idx"].eq(steering_df["clean_prediction_idx"])
steering_df["broken_case_rescued"] = steering_df["clean_correct_broken"] & steering_df["steered_is_correct"]

clean_steering_frames = []
for steering_method, steering_direction in steering_directions.items():
    for steering_scale in STEERING_SCALES:
        clean_steering_part = run_clean_steering_control(
            eval_rows,
            rescue_layer_index,
            INTERVENTION_BATCH_SIZE,
            steering_direction=steering_direction,
            steering_scale=steering_scale,
            desc=f"clean steering control, {steering_method}, scale {steering_scale}",
        ).rename(
            columns={
                "predicted_answer_index": "steered_prediction_idx",
                "true_answer_vs_best_other_logit_gap": "steered_true_answer_vs_best_other_logit_gap",
                "is_correct": "steered_is_correct",
            }
        )
        clean_steering_part["steering_method"] = steering_method
        clean_steering_part["steering_scale"] = float(steering_scale)
        clean_steering_frames.append(clean_steering_part)

clean_steering_df = pd.concat(clean_steering_frames, ignore_index=True)
clean_steering_df = clean_steering_df.merge(
    eval_clean_df[[
        "example_id",
        "clean_prediction_idx",
        "clean_true_answer_vs_best_other_logit_gap",
        "clean_is_correct",
    ]],
    on="example_id",
    how="left",
    validate="many_to_one",
)
clean_steering_df["rescue_layer_number"] = rescue_layer_number
clean_steering_df["delta_true_answer_vs_best_other_logit_gap"] = clean_steering_df["steered_true_answer_vs_best_other_logit_gap"] - clean_steering_df["clean_true_answer_vs_best_other_logit_gap"]
clean_steering_df["prediction_changed"] = ~clean_steering_df["steered_prediction_idx"].eq(clean_steering_df["clean_prediction_idx"])
clean_steering_df["clean_correct_harmed"] = clean_steering_df["clean_is_correct"] & (~clean_steering_df["steered_is_correct"])

display(corrupted_df.head())
display(patch_df.head())
display(steering_df.head())
display(clean_steering_df.head())


## Basic Analysis


Shaded bands in the line plots show standard deviation across the configured corruption scales. They do not represent seed uncertainty.


### Corruption-Only Summary


In [ ]:
corrupted_summary_by_scale = (
    corrupted_df.groupby(["corruption_noise_scale", "corruption_layer_number"])
    .agg(
        mean_corrupted_true_answer_vs_best_other_logit_gap=("corrupted_true_answer_vs_best_other_logit_gap", "mean"),
        mean_delta_true_answer_vs_best_other_logit_gap=("delta_true_answer_vs_best_other_logit_gap", "mean"),
        prediction_preservation_rate=("prediction_preserved", "mean"),
        corrupted_accuracy=("corrupted_is_correct", "mean"),
        clean_correct_break_rate=("clean_correct_broken", "mean"),
    )
    .reset_index()
)

corrupted_summary = (
    corrupted_summary_by_scale.groupby("corruption_layer_number")
    .agg(
        mean_delta_true_answer_vs_best_other_logit_gap_mean=("mean_delta_true_answer_vs_best_other_logit_gap", "mean"),
        mean_delta_true_answer_vs_best_other_logit_gap_std=("mean_delta_true_answer_vs_best_other_logit_gap", "std"),
        prediction_preservation_rate_mean=("prediction_preservation_rate", "mean"),
        prediction_preservation_rate_std=("prediction_preservation_rate", "std"),
        clean_correct_break_rate_mean=("clean_correct_break_rate", "mean"),
        clean_correct_break_rate_std=("clean_correct_break_rate", "std"),
    )
    .reset_index()
    .fillna(0.0)
)

display(corrupted_summary_by_scale.round(4))
display(corrupted_summary.round(4))


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)

x = corrupted_summary["corruption_layer_number"].to_numpy()

axes[0].plot(x, corrupted_summary["mean_delta_true_answer_vs_best_other_logit_gap_mean"], marker="o")
axes[0].fill_between(
    x,
    corrupted_summary["mean_delta_true_answer_vs_best_other_logit_gap_mean"] - corrupted_summary["mean_delta_true_answer_vs_best_other_logit_gap_std"],
    corrupted_summary["mean_delta_true_answer_vs_best_other_logit_gap_mean"] + corrupted_summary["mean_delta_true_answer_vs_best_other_logit_gap_std"],
    alpha=0.2,
)
axes[0].axhline(0.0, color="black", linewidth=1, alpha=0.6)
axes[0].set_ylabel("Mean delta")
axes[0].set_title("Logit Gap Change After Corruption")

axes[1].plot(x, corrupted_summary["prediction_preservation_rate_mean"], marker="o")
axes[1].fill_between(
    x,
    corrupted_summary["prediction_preservation_rate_mean"] - corrupted_summary["prediction_preservation_rate_std"],
    corrupted_summary["prediction_preservation_rate_mean"] + corrupted_summary["prediction_preservation_rate_std"],
    alpha=0.2,
)
axes[1].set_ylabel("Dataset fraction")
axes[1].set_title("Prediction Preservation After Corruption")

axes[2].plot(x, corrupted_summary["clean_correct_break_rate_mean"], marker="o")
axes[2].fill_between(
    x,
    corrupted_summary["clean_correct_break_rate_mean"] - corrupted_summary["clean_correct_break_rate_std"],
    corrupted_summary["clean_correct_break_rate_mean"] + corrupted_summary["clean_correct_break_rate_std"],
    alpha=0.2,
)
axes[2].set_xlabel("Corruption layer")
axes[2].set_ylabel("Dataset fraction")
axes[2].set_title("Clean-Correct Break Rate After Corruption")

plt.tight_layout()
plt.show()


### Fixed-Site Same-Example Clean Patching Summary


In [ ]:
patch_summary_by_scale = (
    patch_df.groupby(["corruption_noise_scale", "corruption_layer_number"])
    .agg(
        mean_patched_true_answer_vs_best_other_logit_gap=("patched_true_answer_vs_best_other_logit_gap", "mean"),
        mean_recovered_true_answer_vs_best_other_logit_gap=("recovered_true_answer_vs_best_other_logit_gap", "mean"),
        patch_prediction_matches_clean_rate=("prediction_matches_clean", "mean"),
        patched_accuracy=("patched_is_correct", "mean"),
        broken_case_rescue_rate=("broken_case_rescued", "mean"),
    )
    .reset_index()
)

patch_summary = (
    patch_summary_by_scale.groupby("corruption_layer_number")
    .agg(
        mean_recovered_true_answer_vs_best_other_logit_gap_mean=("mean_recovered_true_answer_vs_best_other_logit_gap", "mean"),
        mean_recovered_true_answer_vs_best_other_logit_gap_std=("mean_recovered_true_answer_vs_best_other_logit_gap", "std"),
        patch_prediction_matches_clean_rate_mean=("patch_prediction_matches_clean_rate", "mean"),
        patch_prediction_matches_clean_rate_std=("patch_prediction_matches_clean_rate", "std"),
        patched_accuracy_mean=("patched_accuracy", "mean"),
        patched_accuracy_std=("patched_accuracy", "std"),
        broken_case_rescue_rate_mean=("broken_case_rescue_rate", "mean"),
        broken_case_rescue_rate_std=("broken_case_rescue_rate", "std"),
    )
    .reset_index()
    .fillna(0.0)
)

display(patch_summary_by_scale.round(4))
display(patch_summary.round(4))


In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 16), sharex=True)

x = patch_summary["corruption_layer_number"].to_numpy()

axes[0].plot(x, patch_summary["mean_recovered_true_answer_vs_best_other_logit_gap_mean"], marker="o")
axes[0].fill_between(
    x,
    patch_summary["mean_recovered_true_answer_vs_best_other_logit_gap_mean"] - patch_summary["mean_recovered_true_answer_vs_best_other_logit_gap_std"],
    patch_summary["mean_recovered_true_answer_vs_best_other_logit_gap_mean"] + patch_summary["mean_recovered_true_answer_vs_best_other_logit_gap_std"],
    alpha=0.2,
)
axes[0].axhline(0.0, color="black", linewidth=1, alpha=0.6)
axes[0].set_ylabel("Mean recovery")
axes[0].set_title("Logit Gap Recovery After Clean Patch")

axes[1].plot(x, patch_summary["patch_prediction_matches_clean_rate_mean"], marker="o")
axes[1].fill_between(
    x,
    patch_summary["patch_prediction_matches_clean_rate_mean"] - patch_summary["patch_prediction_matches_clean_rate_std"],
    patch_summary["patch_prediction_matches_clean_rate_mean"] + patch_summary["patch_prediction_matches_clean_rate_std"],
    alpha=0.2,
)
axes[1].set_ylabel("Dataset fraction")
axes[1].set_title("Prediction Match Rate After Clean Patch")

axes[2].plot(x, patch_summary["patched_accuracy_mean"], marker="o")
axes[2].fill_between(
    x,
    patch_summary["patched_accuracy_mean"] - patch_summary["patched_accuracy_std"],
    patch_summary["patched_accuracy_mean"] + patch_summary["patched_accuracy_std"],
    alpha=0.2,
)
axes[2].set_ylabel("Dataset fraction")
axes[2].set_title("Accuracy After Clean Patch")

axes[3].plot(x, patch_summary["broken_case_rescue_rate_mean"], marker="o")
axes[3].fill_between(
    x,
    patch_summary["broken_case_rescue_rate_mean"] - patch_summary["broken_case_rescue_rate_std"],
    patch_summary["broken_case_rescue_rate_mean"] + patch_summary["broken_case_rescue_rate_std"],
    alpha=0.2,
)
axes[3].set_xlabel("Earlier corruption layer")
axes[3].set_ylabel("Dataset fraction")
axes[3].set_title("Rescue Rate After Clean Patch")

plt.tight_layout()
plt.show()


### Fixed-Site Steering Summary

steering_scale is the multiplier applied to the normalized steering direction at the rescue layer.


In [ ]:
steering_summary_by_scale = (
    steering_df.groupby(["steering_method", "steering_scale", "corruption_noise_scale", "corruption_layer_number"])
    .agg(
        mean_steered_true_answer_vs_best_other_logit_gap=("steered_true_answer_vs_best_other_logit_gap", "mean"),
        mean_recovered_true_answer_vs_best_other_logit_gap=("recovered_true_answer_vs_best_other_logit_gap", "mean"),
        steering_prediction_matches_clean_rate=("prediction_matches_clean", "mean"),
        steered_accuracy=("steered_is_correct", "mean"),
        broken_case_rescue_rate=("broken_case_rescued", "mean"),
    )
    .reset_index()
)

steering_summary = (
    steering_summary_by_scale.groupby(["steering_method", "steering_scale", "corruption_layer_number"])
    .agg(
        mean_recovered_true_answer_vs_best_other_logit_gap_mean=("mean_recovered_true_answer_vs_best_other_logit_gap", "mean"),
        mean_recovered_true_answer_vs_best_other_logit_gap_std=("mean_recovered_true_answer_vs_best_other_logit_gap", "std"),
        steered_accuracy_mean=("steered_accuracy", "mean"),
        steered_accuracy_std=("steered_accuracy", "std"),
        broken_case_rescue_rate_mean=("broken_case_rescue_rate", "mean"),
        broken_case_rescue_rate_std=("broken_case_rescue_rate", "std"),
    )
    .reset_index()
    .fillna(0.0)
)

steering_average_summary = (
    steering_summary.groupby(["steering_method", "steering_scale"])[
        ["mean_recovered_true_answer_vs_best_other_logit_gap_mean", "steered_accuracy_mean", "broken_case_rescue_rate_mean"]
    ]
    .mean()
    .reset_index()
)

display(steering_summary_by_scale.round(4))
display(steering_summary.round(4))
display(steering_average_summary.round(4))


In [ ]:
methods = steering_summary["steering_method"].drop_duplicates().tolist()
fig, axes = plt.subplots(len(methods), 2, figsize=(16, 5 * len(methods)), sharex=True)
if len(methods) == 1:
    axes = np.array([axes])

for row_idx, method in enumerate(methods):
    part = steering_summary.loc[steering_summary["steering_method"].eq(method)].sort_values(["steering_scale", "corruption_layer_number"])
    for scale in sorted(part["steering_scale"].unique()):
        scale_part = part.loc[part["steering_scale"].eq(scale)].sort_values("corruption_layer_number")
        x = scale_part["corruption_layer_number"].to_numpy()
        axes[row_idx, 0].plot(
            x,
            scale_part["mean_recovered_true_answer_vs_best_other_logit_gap_mean"],
            marker="o",
            label=f"steering scale={scale:g}",
        )
        axes[row_idx, 0].fill_between(
            x,
            scale_part["mean_recovered_true_answer_vs_best_other_logit_gap_mean"] - scale_part["mean_recovered_true_answer_vs_best_other_logit_gap_std"],
            scale_part["mean_recovered_true_answer_vs_best_other_logit_gap_mean"] + scale_part["mean_recovered_true_answer_vs_best_other_logit_gap_std"],
            alpha=0.2,
        )
        axes[row_idx, 1].plot(
            x,
            scale_part["broken_case_rescue_rate_mean"],
            marker="o",
            label=f"steering scale={scale:g}",
        )
        axes[row_idx, 1].fill_between(
            x,
            scale_part["broken_case_rescue_rate_mean"] - scale_part["broken_case_rescue_rate_std"],
            scale_part["broken_case_rescue_rate_mean"] + scale_part["broken_case_rescue_rate_std"],
            alpha=0.2,
        )

    axes[row_idx, 0].axhline(0.0, color="black", linewidth=1, alpha=0.6)
    axes[row_idx, 0].set_ylabel("Mean recovery")
    axes[row_idx, 0].set_title(f"{method}: Logit Gap Recovery")
    axes[row_idx, 0].legend()

    axes[row_idx, 1].set_ylabel("Dataset fraction")
    axes[row_idx, 1].set_title(f"{method}: Rescue Rate")
    axes[row_idx, 1].legend()

axes[-1, 0].set_xlabel("Earlier corruption layer")
axes[-1, 1].set_xlabel("Earlier corruption layer")
plt.tight_layout()
plt.show()


In [ ]:
display(
    patch_summary.sort_values(
        ["broken_case_rescue_rate_mean", "mean_recovered_true_answer_vs_best_other_logit_gap_mean"],
        ascending=[False, False],
    )
    .head(10)
    .round(4)
)

display(
    steering_average_summary.sort_values(
        ["broken_case_rescue_rate_mean", "mean_recovered_true_answer_vs_best_other_logit_gap_mean"],
        ascending=[False, False],
    )
    .round(4)
)


### Clean Steering Control

This control applies steering at the rescue layer without any earlier corruption. The purpose is to measure harm on the clean trajectory.


In [ ]:
clean_steering_summary = (
    clean_steering_df.groupby(["steering_method", "steering_scale"])
    .agg(
        mean_delta_true_answer_vs_best_other_logit_gap=("delta_true_answer_vs_best_other_logit_gap", "mean"),
        steered_accuracy=("steered_is_correct", "mean"),
        prediction_change_rate=("prediction_changed", "mean"),
        clean_correct_harm_rate=("clean_correct_harmed", "mean"),
    )
    .reset_index()
)

display(clean_steering_summary.round(4))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for method in clean_steering_summary["steering_method"].drop_duplicates():
    part = clean_steering_summary.loc[clean_steering_summary["steering_method"].eq(method)].sort_values("steering_scale")
    x = part["steering_scale"].to_numpy()
    axes[0].plot(x, part["steered_accuracy"], marker="o", label=method)
    axes[1].plot(x, part["clean_correct_harm_rate"], marker="o", label=method)

axes[0].set_xlabel("Steering scale")
axes[0].set_ylabel("Dataset fraction")
axes[0].set_title("Clean-Run Accuracy Under Steering")
axes[0].legend()

axes[1].set_xlabel("Steering scale")
axes[1].set_ylabel("Dataset fraction")
axes[1].set_title("Clean-Correct Harm Rate Under Steering")
axes[1].legend()

plt.tight_layout()
plt.show()


## Single-Scale Rescue-Layer Sweep

This section scans all rescue layers against all earlier corruption layers at one corruption scale and one steering scale. The purpose is to map rescue structure without repeating manual notebook runs.


In [ ]:
sweep_corruption_noise_scale = 0.25
sweep_steering_scale = 1.0
sweep_steering_methods = ["Contrastive mean direction", "Probe-normal direction"]


In [ ]:
def extract_all_layer_outputs(frame, batch_size, desc):
    rows = []
    hidden_by_layer = {layer_number: [] for layer_number in range(1, L + 1)}

    for start in tqdm(range(0, len(frame), batch_size), total=int(math.ceil(len(frame) / batch_size)), desc=desc):
        batch_df = frame.iloc[start:start + batch_size].reset_index(drop=True)
        batch_cpu = encode_prompts(batch_df["text"], tok, MAX_SEQ_LEN)
        decision_pos = batch_cpu.pop("decision_pos")
        batch = {k: v.to(input_device) for k, v in batch_cpu.items()}
        decision_pos = decision_pos.to(input_device)
        true_choice_idx = torch.tensor([LETTERS.index(str(x)) for x in batch_df["answerKey"].tolist()], dtype=torch.long, device=input_device)

        with torch.inference_mode():
            out = model(**batch, return_dict=True, use_cache=False, output_hidden_states=True)

        choice_logits = select_choice_logits(out.logits, decision_pos)
        predicted_answer_index, true_answer_vs_best_other_logit_gap = compute_choice_metrics(choice_logits, true_choice_idx)
        clean_is_correct = predicted_answer_index.eq(true_choice_idx.detach().cpu())

        row_idx = torch.arange(len(batch_df), device=decision_pos.device)
        for layer_number in range(1, L + 1):
            hidden = out.hidden_states[layer_number][row_idx, decision_pos].detach().cpu().to(torch.float32)
            hidden_by_layer[layer_number].append(hidden)

        for i, row in batch_df.iterrows():
            rows.append(
                {
                    "example_id": row["example_id"],
                    "clean_prediction_idx": int(predicted_answer_index[i].item()),
                    "clean_true_answer_vs_best_other_logit_gap": float(true_answer_vs_best_other_logit_gap[i].item()),
                    "clean_is_correct": bool(clean_is_correct[i].item()),
                }
            )

    hidden_by_layer = {layer_number: torch.cat(blocks, dim=0) for layer_number, blocks in hidden_by_layer.items()}
    return pd.DataFrame(rows), hidden_by_layer


sweep_train_clean_df, sweep_train_hidden_by_layer = extract_all_layer_outputs(
    train_rows,
    CLEAN_BATCH_SIZE,
    desc="sweep train layer extraction",
)

sweep_eval_clean_df, sweep_eval_hidden_by_layer = extract_all_layer_outputs(
    eval_rows,
    CLEAN_BATCH_SIZE,
    desc="sweep eval layer extraction",
)

sweep_corrupted_df = run_fixed_site_intervention_scan(
    eval_rows,
    list(range(0, L - 1)),
    0,
    float(sweep_corruption_noise_scale),
    INTERVENTION_BATCH_SIZE,
    intervention_kind="corruption_only",
    desc=f"sweep corruption only, scale {sweep_corruption_noise_scale}",
).rename(
    columns={
        "predicted_answer_index": "corrupted_prediction_idx",
        "true_answer_vs_best_other_logit_gap": "corrupted_true_answer_vs_best_other_logit_gap",
        "is_correct": "corrupted_is_correct",
    }
)

sweep_corrupted_df = sweep_corrupted_df.merge(
    sweep_eval_clean_df,
    on="example_id",
    how="left",
    validate="many_to_one",
)
sweep_corrupted_df["corruption_layer_number"] = sweep_corrupted_df["corruption_layer"] + 1
sweep_corrupted_df["clean_correct_broken"] = sweep_corrupted_df["clean_is_correct"] & (~sweep_corrupted_df["corrupted_is_correct"])

sweep_patch_rows = []
sweep_steering_rows = []

for rescue_layer_number in tqdm(range(2, L + 1), desc="rescue layer sweep"):
    rescue_layer_index = rescue_layer_number - 1
    corruption_layer_indices = list(range(0, rescue_layer_index))
    eval_rescue_hidden = sweep_eval_hidden_by_layer[rescue_layer_number]
    train_rescue_hidden = sweep_train_hidden_by_layer[rescue_layer_number]

    contrastive_direction, _ = build_contrastive_mean_direction(
        train_rescue_hidden,
        sweep_train_clean_df["clean_is_correct"],
    )
    probe_direction, _ = build_probe_normal_direction(
        train_rescue_hidden,
        sweep_train_clean_df["clean_is_correct"],
    )
    local_steering_directions = {
        "Contrastive mean direction": contrastive_direction,
        "Probe-normal direction": probe_direction,
    }

    patch_raw = run_fixed_site_intervention_scan(
        eval_rows,
        corruption_layer_indices,
        rescue_layer_index,
        float(sweep_corruption_noise_scale),
        INTERVENTION_BATCH_SIZE,
        intervention_kind="clean_patch",
        clean_rescue_hidden=eval_rescue_hidden,
        desc=f"sweep clean patch, rescue layer {rescue_layer_number}",
    ).rename(
        columns={
            "predicted_answer_index": "patched_prediction_idx",
            "true_answer_vs_best_other_logit_gap": "patched_true_answer_vs_best_other_logit_gap",
            "is_correct": "patched_is_correct",
        }
    )

    patch_joined = patch_raw.merge(
        sweep_corrupted_df[[
            "example_id",
            "corruption_layer",
            "corruption_layer_number",
            "corrupted_true_answer_vs_best_other_logit_gap",
            "clean_correct_broken",
        ]],
        on=["example_id", "corruption_layer"],
        how="left",
        validate="one_to_one",
    )
    patch_joined["recovered_true_answer_vs_best_other_logit_gap"] = (
        patch_joined["patched_true_answer_vs_best_other_logit_gap"] - patch_joined["corrupted_true_answer_vs_best_other_logit_gap"]
    )
    patch_joined["broken_case_rescued"] = patch_joined["clean_correct_broken"] & patch_joined["patched_is_correct"]

    patch_summary_local = (
        patch_joined.groupby("corruption_layer_number")
        .agg(
            mean_recovered_true_answer_vs_best_other_logit_gap=("recovered_true_answer_vs_best_other_logit_gap", "mean"),
            broken_case_rescue_rate=("broken_case_rescued", "mean"),
        )
        .reset_index()
    )
    patch_summary_local["rescue_layer_number"] = rescue_layer_number
    sweep_patch_rows.append(patch_summary_local)

    for steering_method in sweep_steering_methods:
        steering_raw = run_fixed_site_intervention_scan(
            eval_rows,
            corruption_layer_indices,
            rescue_layer_index,
            float(sweep_corruption_noise_scale),
            INTERVENTION_BATCH_SIZE,
            intervention_kind="steering",
            steering_direction=local_steering_directions[steering_method],
            steering_scale=float(sweep_steering_scale),
            desc=f"sweep steering, {steering_method}, rescue layer {rescue_layer_number}",
        ).rename(
            columns={
                "predicted_answer_index": "steered_prediction_idx",
                "true_answer_vs_best_other_logit_gap": "steered_true_answer_vs_best_other_logit_gap",
                "is_correct": "steered_is_correct",
            }
        )

        steering_joined = steering_raw.merge(
            sweep_corrupted_df[[
                "example_id",
                "corruption_layer",
                "corruption_layer_number",
                "corrupted_true_answer_vs_best_other_logit_gap",
                "clean_correct_broken",
            ]],
            on=["example_id", "corruption_layer"],
            how="left",
            validate="one_to_one",
        )
        steering_joined["recovered_true_answer_vs_best_other_logit_gap"] = (
            steering_joined["steered_true_answer_vs_best_other_logit_gap"] - steering_joined["corrupted_true_answer_vs_best_other_logit_gap"]
        )
        steering_joined["broken_case_rescued"] = steering_joined["clean_correct_broken"] & steering_joined["steered_is_correct"]

        steering_summary_local = (
            steering_joined.groupby("corruption_layer_number")
            .agg(
                mean_recovered_true_answer_vs_best_other_logit_gap=("recovered_true_answer_vs_best_other_logit_gap", "mean"),
                broken_case_rescue_rate=("broken_case_rescued", "mean"),
            )
            .reset_index()
        )
        steering_summary_local["rescue_layer_number"] = rescue_layer_number
        steering_summary_local["steering_method"] = steering_method
        sweep_steering_rows.append(steering_summary_local)

sweep_patch_summary = pd.concat(sweep_patch_rows, ignore_index=True)
sweep_steering_summary = pd.concat(sweep_steering_rows, ignore_index=True)

display(sweep_patch_summary.head())
display(sweep_steering_summary.head())


In [ ]:
patch_recovery_heatmap = sweep_patch_summary.pivot(
    index="corruption_layer_number",
    columns="rescue_layer_number",
    values="mean_recovered_true_answer_vs_best_other_logit_gap",
)
patch_rescue_heatmap = sweep_patch_summary.pivot(
    index="corruption_layer_number",
    columns="rescue_layer_number",
    values="broken_case_rescue_rate",
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

im0 = axes[0].imshow(patch_recovery_heatmap, origin="lower", aspect="auto", cmap="viridis")
axes[0].set_title("Clean Patch: Logit Gap Recovery")
axes[0].set_xlabel("Rescue layer")
axes[0].set_ylabel("Corruption layer")
axes[0].set_xticks(range(patch_recovery_heatmap.shape[1]))
axes[0].set_xticklabels(patch_recovery_heatmap.columns.tolist(), rotation=90)
axes[0].set_yticks(range(patch_recovery_heatmap.shape[0]))
axes[0].set_yticklabels(patch_recovery_heatmap.index.tolist())
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(patch_rescue_heatmap, origin="lower", aspect="auto", cmap="magma")
axes[1].set_title("Clean Patch: Rescue Rate")
axes[1].set_xlabel("Rescue layer")
axes[1].set_ylabel("Corruption layer")
axes[1].set_xticks(range(patch_rescue_heatmap.shape[1]))
axes[1].set_xticklabels(patch_rescue_heatmap.columns.tolist(), rotation=90)
axes[1].set_yticks(range(patch_rescue_heatmap.shape[0]))
axes[1].set_yticklabels(patch_rescue_heatmap.index.tolist())
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


In [ ]:
methods = sweep_steering_summary["steering_method"].drop_duplicates().tolist()
fig, axes = plt.subplots(len(methods), 2, figsize=(16, 6 * len(methods)))
if len(methods) == 1:
    axes = np.array([axes])

for row_idx, method in enumerate(methods):
    part = sweep_steering_summary.loc[sweep_steering_summary["steering_method"].eq(method)]
    recovery_heatmap = part.pivot(
        index="corruption_layer_number",
        columns="rescue_layer_number",
        values="mean_recovered_true_answer_vs_best_other_logit_gap",
    )
    rescue_heatmap = part.pivot(
        index="corruption_layer_number",
        columns="rescue_layer_number",
        values="broken_case_rescue_rate",
    )

    im0 = axes[row_idx, 0].imshow(recovery_heatmap, origin="lower", aspect="auto", cmap="viridis")
    axes[row_idx, 0].set_title(f"{method}: Logit Gap Recovery")
    axes[row_idx, 0].set_xlabel("Rescue layer")
    axes[row_idx, 0].set_ylabel("Corruption layer")
    axes[row_idx, 0].set_xticks(range(recovery_heatmap.shape[1]))
    axes[row_idx, 0].set_xticklabels(recovery_heatmap.columns.tolist(), rotation=90)
    axes[row_idx, 0].set_yticks(range(recovery_heatmap.shape[0]))
    axes[row_idx, 0].set_yticklabels(recovery_heatmap.index.tolist())
    fig.colorbar(im0, ax=axes[row_idx, 0], fraction=0.046, pad=0.04)

    im1 = axes[row_idx, 1].imshow(rescue_heatmap, origin="lower", aspect="auto", cmap="magma")
    axes[row_idx, 1].set_title(f"{method}: Rescue Rate")
    axes[row_idx, 1].set_xlabel("Rescue layer")
    axes[row_idx, 1].set_ylabel("Corruption layer")
    axes[row_idx, 1].set_xticks(range(rescue_heatmap.shape[1]))
    axes[row_idx, 1].set_xticklabels(rescue_heatmap.columns.tolist(), rotation=90)
    axes[row_idx, 1].set_yticks(range(rescue_heatmap.shape[0]))
    axes[row_idx, 1].set_yticklabels(rescue_heatmap.index.tolist())
    fig.colorbar(im1, ax=axes[row_idx, 1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()
